# NILM – Semaine 1 : prise en main des données UK-DALE

**Objectif de la semaine.** Avant de construire un modèle, je dois connaître les données : charger un jeu de mesures de consommation (UK-DALE ou REDD), regarder la courbe de la maison entière et celles des appareils, et identifier les appareils qui pèsent le plus dans la consommation. C'est cette courbe totale que le NILM (*Non-Intrusive Load Monitoring*) devra ensuite « démonter » en appareils, à partir d'un seul capteur placé au tableau électrique.

**Outils.** Je n'utilise que `pandas`, `numpy` et `matplotlib`. J'ai volontairement laissé de côté NILMTK, dont l'installation pose souvent des conflits de versions : je relis directement les fichiers bruts, ce qui me donne aussi la maîtrise complète du pré-traitement.

**Données utilisées.** UK-DALE, maison 1 : une maison britannique mesurée de fin 2012 à début 2015, avec un compteur sur le total (`aggregate`) et un compteur par appareil, un point toutes les 6 secondes. C'est un jeu public, mesuré appareil par appareil (je connais donc la « vraie réponse » à comparer) et au même format que REDD. Pour cette première exploration, je charge les 15 premiers jours de mars 2013 et cinq canaux : la maison entière, le lave-linge, le lave-vaisselle, le frigo et la bouilloire.

Le code garde le mode démo de la première version (données synthétiques si `DATA_DIR` est absent), mais toutes les analyses ci-dessous portent sur les vraies mesures.

## 1. Configuration
Les fichiers lus sont ceux du téléchargement brut d'UK-DALE : un dossier par maison, des fichiers `channel_N.dat` avec une ligne par mesure (`horodatage_unix puissance`) et un fichier `labels.dat` qui associe chaque numéro de canal à un nom (`N nom`).

Réglages principaux :
- `DATA_DIR` : dossier de la maison 1 (sous Colab, monté depuis Google Drive) ;
- `START`, `END` : période chargée (du 1er au 15 mars 2013 ; charger les deux ans complets serait inutilement lourd pour une exploration) ;
- `CHANNELS` : les canaux retenus (`aggregate`, `washing_machine`, `dishwasher`, `fridge`, `kettle`) ;
- `RESAMPLE` : pas d'une minute.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- Google Colab : connecte ton Drive (ignoré hors Colab) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Hors Colab : pas de montage de Drive.")

# ================== RÉGLAGES (à adapter) ==================
DATA_DIR = Path("/content/drive/MyDrive/nilm/house_1")   # dossier d'une maison UK-DALE : channel_N.dat + labels.dat
START, END = "2013-03-01", "2013-03-15"                  # période chargée ; None, None = tout (lourd !)
CHANNELS = ["aggregate", "washing_machine", "dishwasher", "fridge", "kettle"]   # None = tous les canaux
# ===========================================================

MAINS_NAMES = ("mains", "aggregate", "site_meter")   # noms de canaux considérés comme "maison entière"
RESAMPLE = "1min"                 # pas de temps pour l'exploration
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True

## 2. Données de démonstration (utilisées seulement si `DATA_DIR` est absent)
La fonction `make_demo` écrit 3 jours d'une maison fictive au pas de 6 s : une veille d'environ 60 W, un frigo qui cycle toutes les 15 minutes, un chauffe-eau de 2,1 kW la nuit et à midi, et un lave-linge chaque soir (chauffe puis rotation). Cela suffit pour vérifier que la lecture, les graphiques et les bilans fonctionnent.

In [ ]:
def make_demo(path, days=3, step_s=6, seed=0):
    """Écrit une fausse maison au même format que REDD / UK-DALE (données SYNTHÉTIQUES)."""
    rng = np.random.default_rng(seed)
    path.mkdir(parents=True, exist_ok=True)
    t = np.arange(0, days * 86400, step_s)
    hour = (t / 3600) % 24
    base = 60 + rng.normal(0, 3, t.size)                                   # veille permanente
    fridge = np.where(((t // 900) % 2 == 0), 90.0, 0.0)                    # cycles de ~15 min
    heater = np.where(((hour > 2) & (hour < 4.5)) | ((hour > 12) & (hour < 13)), 2100.0, 0.0)   # chauffe-eau
    washer = np.zeros_like(t, dtype=float)
    for d in range(days):
        s = d * 86400 + 19.5 * 3600
        washer[(t >= s) & (t < s + 3600)] = 500 + 1800 * ((t[(t >= s) & (t < s + 3600)] - s) < 1200)
    ts0 = 1_700_000_000
    chans = {"mains": base + fridge + heater + washer, "fridge": fridge + 5, "water_heater": heater, "washing_machine": washer}
    labels = []
    for i, (name, p) in enumerate(chans.items(), start=1):
        pd.DataFrame({"t": ts0 + t, "w": np.maximum(p, 0).round(1)}).to_csv(path / f"channel_{i}.dat", sep=" ", header=False, index=False)
        labels.append(f"{i} {name}")
    (path / "labels.dat").write_text("\n".join(labels) + "\n")

if not DATA_DIR.exists():
    print("DATA_DIR introuvable : génération de données de DÉMO synthétiques.")
    DATA_DIR = Path("demo_house")
    START, END, CHANNELS = None, None, None      # le jeu de démo a d'autres noms de canaux
    make_demo(DATA_DIR)
print("Dossier utilisé :", DATA_DIR.resolve())

## 3. Chargement
Je lis `labels.dat`, puis chaque `channel_N.dat` : la première colonne (horodatage Unix) devient l'index temporel, la deuxième la puissance. Je supprime les doublons d'horodatage, je trie, puis je rééchantillonne chaque canal à la minute (moyenne) pour les aligner dans un seul tableau.

In [ ]:
def read_labels(folder):
    labels = {}
    for line in (folder / "labels.dat").read_text().splitlines():
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            labels[int(parts[0])] = parts[1].strip()
    return labels

def read_channel(folder, ch):
    """Lit channel_N.dat : colonne 0 = horodatage unix, colonne 1 = puissance (W)."""
    df = pd.read_csv(folder / f"channel_{ch}.dat", sep=r"\s+", header=None, usecols=[0, 1], names=["t", "w"])
    df.index = pd.to_datetime(df["t"], unit="s")
    s = df["w"].astype(float)
    s = s[~s.index.duplicated()].sort_index()
    if START or END:
        s = s.loc[START:END]
    return s

labels = read_labels(DATA_DIR)
print(len(labels), "canaux dans labels.dat")
series = {}
for ch, name in labels.items():
    if CHANNELS is not None and name not in CHANNELS:
        continue
    try:
        s = read_channel(DATA_DIR, ch)
    except FileNotFoundError:
        print("fichier manquant :", f"channel_{ch}.dat", f"({name})")
        continue
    if s.empty:
        print("AUCUNE donnée sur la période demandée pour", name, "-> change START/END (ou mets None, None)")
        continue
    series[name] = s.resample(RESAMPLE).mean()
    print(f"chargé : channel_{ch}.dat ({name}) {s.index.min()} -> {s.index.max()}")
if not series:
    raise SystemExit("Aucune donnée chargée : vérifie DATA_DIR, START/END et CHANNELS.")
data = pd.DataFrame(series)
print(data.shape, "|", data.index.min(), "->", data.index.max())
data.head()

## 4. La maison entière : la courbe qu'on va devoir « démonter »
C'est la seule information dont disposera le capteur dans le projet final.

> **Remarque importante sur UK-DALE** : le canal `aggregate` est mesuré par une pince ampèremétrique et donne une puissance **apparente** (VA), alors que les prises des appareils mesurent une puissance **active** (W). Pour explorer, c'est suffisant. J'ai finalement gardé ce canal pour l'entraînement : c'est aussi ce que mesurera le capteur de courant (CT) du projet, qui ne voit que le courant, et ce qui se rapproche de la puissance apparente donnée par la TIC du Linky (`SINSTS`).

In [ ]:
mains_cols = [c for c in data.columns if c.lower() in MAINS_NAMES or c.lower().startswith("mains")]
print("Canaux maison entière :", mains_cols)
mains = data[mains_cols].sum(axis=1) if len(mains_cols) > 1 else data[mains_cols[0]]   # REDD : 2 phases à additionner
ax = mains.plot(title="Puissance de la maison entière (W)", color="#0E7C86")
ax.set_ylabel("W"); plt.show()

## 5. Un jour, maison entière et appareils
Sur une journée complète, je superpose la courbe de la maison et celles des 3 appareils qui consomment le plus d'énergie. On voit déjà que chaque appareil laisse une « signature » dans la courbe totale (niveau de puissance, durée, répétition), mais qu'elles se mélangent avec le reste de la maison.

In [ ]:
appliances = [c for c in data.columns if c not in mains_cols]
day = data.index.min().normalize() + pd.Timedelta(days=1)       # un jour complet (le 2e)
d = data.loc[day: day + pd.Timedelta(days=1)]
fig, axes = plt.subplots(1 + min(3, len(appliances)), 1, figsize=(11, 8), sharex=True)
axes[0].plot(d.index, d[mains_cols].sum(axis=1), color="#0E7C86"); axes[0].set_title("Maison entière (W)")
top = data[appliances].sum().sort_values(ascending=False).index[:3]   # les 3 plus gros en énergie
for ax, name in zip(axes[1:], top):
    ax.plot(d.index, d[name], color="#E8A000"); ax.set_title(name + " (W)")
plt.tight_layout(); plt.show()

## 6. Bilan d'énergie : quels appareils compter ?
Pour la flexibilité, ce qui compte, ce sont les gros consommateurs que l'on peut décaler dans le temps. Ce tableau donne, pour chaque canal, l'énergie consommée (kWh), sa part de la maison, sa puissance maximale et la part du temps où il est allumé (plus de 10 W).

In [ ]:
hours = pd.Timedelta(RESAMPLE).total_seconds() / 3600
energy_kwh = (data.fillna(0) * hours).sum() / 1000
summary = pd.DataFrame({"kWh": energy_kwh})
summary["part de la maison (%)"] = 100 * summary["kWh"] / energy_kwh[mains_cols].sum()
summary["puissance max (W)"] = data.max()
summary["% du temps allumé (>10 W)"] = 100 * (data > 10).mean()
summary.sort_values("kWh", ascending=False).round(1)

## 7. Événements : les marches de la courbe
Les méthodes de NILM « classiques » repèrent les sauts de puissance au moment où un appareil s'allume ou s'éteint. Je liste les sauts de plus de 300 W de la courbe totale, puis je trace leur histogramme : chaque pic de l'histogramme correspond à une famille d'appareils de même puissance. Si deux appareils ont la même puissance, ils tombent dans le même pic : c'est la limite de cette approche.

In [ ]:
step = mains.diff()
events = step[step.abs() > 300].dropna()          # seuil en W : à ajuster
print(len(events), "événements > 300 W")
events.sort_values(key=lambda s: -s.abs()).head(10).round(0)

In [ ]:
fig, ax = plt.subplots()
ax.hist(events.values, bins=40, color="#0E7C86")
ax.set_title("Taille des sauts de puissance (W) : chaque pic est une famille d'appareils")
plt.show()

## 8. Ce que l'exploration m'a appris
Sur les 14 premiers jours de mars 2013 :
- la maison consomme **107,8 kWh**, soit environ **7,7 kWh par jour**. La consommation de fond (veilles, éclairage, box…) est de 150 à 250 W, avec des pics de 2 à 4,7 kW ;
- part de l'énergie totale : frigo 9,7 %, lave-linge 7,1 %, bouilloire 5,1 %, lave-vaisselle 5,0 %. **Les quatre appareils suivis ne pèsent que 25 à 27 % du total** : le reste vient d'appareils non suivis (éclairage, TV, four, informatique) ;
- le frigo est allumé environ 35 % du temps (≈ 90 W, avec un pic d'environ 250 W au dégivrage) ; la bouilloire fait des impulsions de 2,4 kW pendant quelques minutes ;
- les sauts de puissance se regroupent en familles autour de ±400-500 W, ±1 000 W et ±2 000-2 400 W. La bouilloire, la chauffe du lave-linge et celle du lave-vaisselle tombent dans la même famille (≈ 2 kW) : un algorithme qui ne regarde que la hauteur du saut les confondra ;
- il y a un trou de mesure autour des 10-11 mars, et les canaux ne s'arrêtent pas tous à la même date : je ne garde que les minutes où **tous** les canaux ont une mesure ;
- **pas de chauffe-eau électrique** dans cette maison (chauffage au gaz) : le chauffe-eau sera **simulé** pour la partie flexibilité.

## 9. Conclusion
- **Données retenues** : UK-DALE maison 1, ramenée à la minute, avec 4 appareils cibles : **frigo, bouilloire, lave-linge, lave-vaisselle**. Ce sont les appareils bien mesurés de cette maison ; le lave-linge et le lave-vaisselle sont aussi les plus intéressants pour la flexibilité (décalables).
- **Entrée du futur modèle** : la seule courbe de puissance totale de la maison (canal `aggregate`, en VA), comme celle que fournira le capteur du projet.
- **Sorties attendues** : pour chaque appareil, sa puissance et son état allumé / éteint, minute par minute.
- **Difficultés identifiées** : 73 à 75 % de l'énergie vient d'appareils non suivis ; trois appareils ont presque la même puissance (≈ 2 kW) ; le frigo (90 W) est plus faible que les variations du fond ; certaines périodes ont des trous de mesure.
- **Intérêt pour la suite** : ces constats orientent directement la semaine 2 (premier modèle simple et mesure de sa précision) et expliquent pourquoi un modèle qui regarde la **forme** de la courbe dans le temps sera nécessaire.